# impact-ledger — Impact Investment Portfolio Tracker
## Demo: CDFI Impact Fund Portfolio Analysis

This notebook demonstrates how to build, analyze, and report on an impact
investment portfolio using impact-ledger.

Use case: A CDFI managing a $45MM portfolio of loans, NMTC deals, and grants
across affordable housing, healthcare, small business, and community facilities.


In [ ]:
import sys
sys.path.insert(0, '..')

from impactledger import (
    loan, nmtc, grant, add_impact, Portfolio,
    cdfi_fund_report, sector_impact_report, watchlist_report,
    INVESTMENT_TYPES, SECTORS
)
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 1. Available Investment Types and Sectors

In [ ]:
print("Investment Types:")
for k, v in INVESTMENT_TYPES.items():
    print(f"  {k:<15} — {v}")

print("\nSectors:")
for k, v in SECTORS.items():
    print(f"  {k:<25} — {v}")


## 2. Build the Portfolio

We create a $45MM portfolio with 8 investments across 4 states.


In [ ]:
p = Portfolio(name="CDFI Impact Fund I", vintage_year=2022)

# ── Healthcare ────────────────────────────────────────────────
hc = loan(
    name="Southside Community Health Center",
    amount=3_500_000,
    interest_rate=0.045,
    date_closed="2022-03-15",
    maturity_date="2032-03-15",
    state="IL",
    borrower_name="Southside Health Inc",
    borrower_type="nonprofit",
    sector="healthcare",
    is_low_income_area=True,
    is_minority_borrower=True,
)
add_impact(hc, jobs_created=32, jobs_retained=48, patients_served=8500)
p.add(hc)

# ── Affordable Housing ────────────────────────────────────────
housing = loan(
    name="Westside Affordable Housing",
    amount=6_000_000,
    interest_rate=0.040,
    date_closed="2022-06-01",
    maturity_date="2037-06-01",
    state="GA",
    borrower_name="Atlanta Housing Partners LLC",
    borrower_type="nonprofit",
    sector="affordable_housing",
    is_low_income_area=True,
    is_minority_borrower=True,
    is_women_borrower=True,
)
add_impact(housing, jobs_created=18, jobs_retained=12, affordable_units=84,
           sq_ft_community_space=12000)
p.add(housing)

# ── NMTC Deal ─────────────────────────────────────────────────
mfg = nmtc(
    name="Detroit Advanced Manufacturing Co",
    amount=9_500_000,
    date_closed="2022-09-01",
    maturity_date="2029-09-01",
    state="MI",
    borrower_name="Detroit Advanced Mfg Co",
    sector="small_business",
    is_low_income_area=True,
    is_minority_borrower=True,
)
add_impact(mfg, jobs_created=52, jobs_retained=78, businesses_supported=1)
p.add(mfg)

# ── Small Business ─────────────────────────────────────────────
sb = loan(
    name="Bronx Small Business Lending Pool",
    amount=4_200_000,
    interest_rate=0.055,
    date_closed="2022-11-15",
    maturity_date="2028-11-15",
    state="NY",
    borrower_name="Bronx Economic Dev Corp",
    borrower_type="nonprofit",
    sector="small_business",
    is_low_income_area=True,
    is_minority_borrower=True,
    is_women_borrower=True,
)
add_impact(sb, jobs_created=95, jobs_retained=120, businesses_supported=42)
p.add(sb)

# ── Community Facility ─────────────────────────────────────────
cf = loan(
    name="Chicago Community Arts Center",
    amount=2_800_000,
    interest_rate=0.042,
    date_closed="2023-01-10",
    maturity_date="2033-01-10",
    state="IL",
    borrower_name="Chicago Arts Foundation",
    borrower_type="nonprofit",
    sector="community_facility",
    is_low_income_area=True,
    is_women_borrower=True,
)
add_impact(cf, jobs_created=14, jobs_retained=22, sq_ft_community_space=18500)
p.add(cf)

# ── Childcare ──────────────────────────────────────────────────
cc = loan(
    name="Harlem Early Learning Center",
    amount=1_800_000,
    interest_rate=0.038,
    date_closed="2023-03-20",
    maturity_date="2030-03-20",
    state="NY",
    borrower_name="Harlem Childcare Inc",
    borrower_type="nonprofit",
    sector="childcare",
    is_low_income_area=True,
    is_minority_borrower=True,
    is_women_borrower=True,
)
add_impact(cc, jobs_created=18, jobs_retained=8, students_served=120)
p.add(cc)

# ── Watch List Investment ──────────────────────────────────────
watch_inv = loan(
    name="Flint Food Co-op",
    amount=950_000,
    interest_rate=0.050,
    date_closed="2021-08-01",
    maturity_date="2028-08-01",
    state="MI",
    borrower_name="Flint Food Cooperative",
    borrower_type="cooperative",
    sector="food_access",
    is_low_income_area=True,
    status="watch",
)
add_impact(watch_inv, jobs_created=8, jobs_retained=5, businesses_supported=1)
p.add(watch_inv)

# ── Grant ──────────────────────────────────────────────────────
g = grant(
    name="Detroit Microenterprise Grant",
    amount=350_000,
    date_closed="2023-05-01",
    state="MI",
    borrower_name="Detroit CDFI Partners",
    borrower_type="cdfi",
    sector="microenterprise",
    is_low_income_area=True,
    is_minority_borrower=True,
)
add_impact(g, jobs_created=12, businesses_supported=28)
p.add(g)

print(f"Portfolio built: {p.count()} investments, ${p.total_committed/1e6:.1f}MM committed")


## 3. Full Portfolio Summary

In [ ]:
p.summary()


## 4. Financial Analysis

In [ ]:
df = p.to_dataframe()

print(f"Total Committed:       ${p.total_committed/1e6:.2f}MM")
print(f"Total Outstanding:     ${p.total_outstanding/1e6:.2f}MM")
print(f"Annual Income:         ${p.annual_income/1e6:.2f}MM")
print(f"Weighted Avg Yield:    {p.weighted_avg_rate*100:.2f}%")
print(f"\nInvestment Type Breakdown:")
print(df.groupby("type")["amount"].agg(["count", "sum"]).to_string())


## 5. Impact Metrics

In [ ]:
print(f"Total Jobs Created:    {p.total_jobs_created:,}")
print(f"Total Jobs Retained:   {p.total_jobs_retained:,}")
print(f"Total Jobs:            {p.total_jobs:,}")
print(f"Affordable Units:      {p.total_affordable_units:,}")
print(f"Community Sq Ft:       {p.total_sq_ft:,.0f}")
print(f"Cost per Job:          ${p.cost_per_job:,.0f}")
print(f"\nDemographics:")
print(f"  % Low-Income Area:   {p.pct_low_income_area*100:.1f}%")
print(f"  % Minority Borrower: {p.pct_minority_borrower*100:.1f}%")
print(f"  % Women Borrower:    {p.pct_women_borrower*100:.1f}%")


## 6. Sector Breakdown

In [ ]:
sector_df = p.sector_breakdown()
print("Sector Breakdown:")
print(sector_df.to_string(index=False))


## 7. Geographic Breakdown

In [ ]:
state_df = p.state_breakdown()
print("State Breakdown:")
print(state_df.to_string(index=False))


## 8. CDFI Fund Report

In [ ]:
cdfi_report = cdfi_fund_report(p)


## 9. Sector Impact Report

In [ ]:
sector_report = sector_impact_report(p)


## 10. Watchlist Report

In [ ]:
watch_report = watchlist_report(p)


## 11. Filter and Slice

In [ ]:
# Illinois investments only
il = p.filter_state("IL")
print(f"Illinois investments: {len(il)}")
for inv in il:
    print(f"  {inv.name} — ${inv.amount/1e6:.2f}MM")

# Active investments only
active = p.filter_status("active")
print(f"\nActive investments: {len(active)}")

# Loans only
loans = p.filter_type("loan")
print(f"Loans: {len(loans)}")

# Affordable housing
housing = p.filter_sector("affordable_housing")
print(f"Affordable housing deals: {len(housing)}")


## Summary

This notebook demonstrated the full impact-ledger workflow:

1. Building a multi-investment portfolio with loans, NMTC, and grants
2. Tracking social impact metrics per investment
3. Aggregating financial metrics (yield, income, outstanding balance)
4. Computing impact metrics (jobs, units, cost per job)
5. Generating CDFI Fund, sector, and watchlist reports
6. Filtering by state, type, sector, and status

**GitHub:** https://github.com/Jaypatel1511/impact-ledger
**PyPI:** https://pypi.org/project/impact-ledger
